# Tugas 2 - Memahami Data

**Mata Kuliah:** Penambangan Data Kelas A

**Tugas:** Run dan modifikasi implementasi materi "Memahami Data" dari website Pak Mulaab menggunakan dataset IRIS.

**Referensi:** https://mulaab.github.io/datamining/memahami-data/

**Deskripsi Tugas:**
1. Implementasi Statistik Deskriptif (mean, median, modus, Q1-Q3, IQR, variansi, standar deviasi, skewness, boxplot)
2. Implementasi Distribusi Data (distribusi normal, histogram)
3. Implementasi Mengukur Jarak Numerik (Minkowski, Manhattan, Euclidean, Average, Weighted Euclidean, Chord, Mahalanobis, Cosine, Pearson)
4. Implementasi Mengukur Jarak Binary (symmetric/asymmetric dissimilarity, Jaccard)
5. Implementasi Mengukur Jarak Categorical (Overlay Metric, VDM, MRM)
6. Implementasi Mengukur Jarak Ordinal
7. Implementasi Mengukur Jarak Campuran

**Dataset:** IRIS.csv (150 data, 4 atribut numerik, 3 kelas)

## Setup: Upload dan Load Dataset

Upload file `IRIS.csv` ke Colab, atau letakkan di folder yang sama dengan notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import minkowski, cityblock, euclidean, cosine, mahalanobis

# Untuk Google Colab: upload file IRIS.csv
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv("IRIS.csv")
df.head(10)

ModuleNotFoundError: No module named 'pandas'

---
## 1. Distribusi Data

Distribusi data yang paling dikenal adalah **distribusi normal (Gaussian)**.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

Di mana $\sigma$ adalah standar deviasi dan $\mu$ adalah mean.

In [ ]:
# Visualisasi distribusi normal dengan mean=1, variance=0.5
mu = 1
variance = 0.5
sigma = np.sqrt(variance)
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
y = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

plt.figure(figsize=(8, 4))
plt.plot(x, y, 'b-', linewidth=2)
plt.title(f'Distribusi Normal (mean={mu}, variance={variance})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Distribusi data sepal_length dari dataset IRIS
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
df['sepal_length'].hist(bins=15, edgecolor='black', alpha=0.7)
plt.title('Histogram sepal_length')
plt.xlabel('sepal_length')
plt.ylabel('Frekuensi')

plt.subplot(1, 2, 2)
df['sepal_length'].plot(kind='kde', linewidth=2)
plt.title('Density Plot sepal_length')
plt.xlabel('sepal_length')

plt.tight_layout()
plt.show()

---
## 2. Statistik Deskriptif

### 2.1 Ukuran Kecenderungan Terpusat (Mean, Median, Modus)

In [ ]:
kolom = 'sepal_length'

print("=" * 45)
print("STATISTIK DESKRIPTIF:", kolom)
print("=" * 45)
print(f"Jumlah data      : {df[kolom].count()}")
print(f"Rata-rata (Mean) : {df[kolom].mean():.4f}")
print(f"Median (Q2)      : {df[kolom].median():.4f}")

# Modus
modus = df[kolom].mode()
if len(modus) > 0:
    nilai_modus = modus.iloc[0]
    jumlah_modus = (df[kolom] == nilai_modus).sum()
    print(f"Modus            : {nilai_modus} (muncul {jumlah_modus} kali)")

print(f"\nNilai Minimal    : {df[kolom].min()}")
print(f"Q1 (25%)         : {df[kolom].quantile(0.25)}")
print(f"Q2 (50%/Median)  : {df[kolom].quantile(0.5)}")
print(f"Q3 (75%)         : {df[kolom].quantile(0.75)}")
print(f"Nilai Maksimal   : {df[kolom].max()}")

### Hubungan Empiris: mean - mode ≈ 3 × (mean - median)

Untuk data unimodal yang cukup miring.

In [ ]:
mean_val = df[kolom].mean()
median_val = df[kolom].median()
mode_val = df[kolom].mode().iloc[0]

print(f"mean - mode                = {mean_val - mode_val:.4f}")
print(f"3 × (mean - median)        = {3 * (mean_val - median_val):.4f}")
print(f"Selisih (approx)           = {abs((mean_val - mode_val) - 3*(mean_val - median_val)):.4f}")

### 2.2 Mengukur Sebaran Data

In [ ]:
Q1 = df[kolom].quantile(0.25)
Q3 = df[kolom].quantile(0.75)
IQR = Q3 - Q1

print("=" * 45)
print("UKURAN SEBARAN DATA")
print("=" * 45)
print(f"Range (Rentang)      : {df[kolom].max() - df[kolom].min():.2f}")
print(f"IQR (Q3 - Q1)        : {IQR:.2f}")
print(f"Variansi             : {df[kolom].var():.4f}")
print(f"Standar Deviasi      : {df[kolom].std():.4f}")
print(f"Skewness             : {df[kolom].skew():.6f}")

# Deteksi outlier
batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR
outliers = df[(df[kolom] < batas_bawah) | (df[kolom] > batas_atas)]
print(f"\nBatas bawah outlier  : {batas_bawah:.2f}")
print(f"Batas atas outlier   : {batas_atas:.2f}")
print(f"Jumlah outlier       : {len(outliers)}")
if len(outliers) > 0:
    print(f"Nilai outlier        : {outliers[kolom].values}")

### 2.3 Ringkasan Lima Angka & Boxplot

Ringkasan lima angka: **Minimum, Q1, Median, Q3, Maksimum**

In [ ]:
# Ringkasan lima angka
print("Ringkasan Lima Angka:")
print(f"  Minimum : {df[kolom].min()}")
print(f"  Q1      : {Q1}")
print(f"  Median  : {df[kolom].median()}")
print(f"  Q3      : {Q3}")
print(f"  Maximum : {df[kolom].max()}")

# Boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot satu kolom
axes[0].boxplot(df[kolom].dropna(), vert=True)
axes[0].set_title(f'Boxplot {kolom}')
axes[0].set_ylabel(kolom)

# Boxplot semua kolom numerik
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
axes[1].boxplot([df[c].dropna() for c in numeric_cols], labels=numeric_cols)
axes[1].set_title('Boxplot Semua Atribut Numerik')

plt.tight_layout()
plt.show()

### 2.4 Skewness (Kemencengan)

Koefisien Kemencengan Pearson:

$$s_k = \frac{\bar{X} - M_o}{s} \approx \frac{3(\bar{X} - M_e)}{s}$$

- $s_k > 0$: Positive skew (menceng kanan)
- $s_k = 0$: Simetris
- $s_k < 0$: Negative skew (menceng kiri)

In [ ]:
mean_val = df[kolom].mean()
median_val = df[kolom].median()
std_val = df[kolom].std()
mode_val = df[kolom].mode().iloc[0]

# Koefisien kemencengan Pearson
sk_mode = (mean_val - mode_val) / std_val
sk_median = 3 * (mean_val - median_val) / std_val

print(f"Skewness (scipy)                     : {df[kolom].skew():.6f}")
print(f"Pearson skewness (mean-mode)/s        : {sk_mode:.6f}")
print(f"Pearson skewness 3(mean-median)/s     : {sk_median:.6f}")

sk = df[kolom].skew()
if sk > 0:
    print("Kesimpulan: Data menceng ke kanan (positive skew)")
elif sk < 0:
    print("Kesimpulan: Data menceng ke kiri (negative skew)")
else:
    print("Kesimpulan: Data simetris")

### 2.5 Statistik Deskriptif Semua Kolom Numerik

In [ ]:
# Ringkasan statistik semua kolom numerik
print(df.describe())
print("\nSkewness per kolom:")
print(df.select_dtypes(include=[np.number]).skew())

---
## 3. Mengukur Jarak Data - Tipe Numerik

Kita akan menggunakan dua vektor contoh dan juga data dari IRIS.

In [ ]:
# Dua vektor contoh
v1 = np.array([1, 2, 3, 4])
v2 = np.array([4, 3, 2, 1])

# Dua baris data IRIS (sebagai contoh)
iris1 = df.iloc[0, :4].values.astype(float)  # Iris-setosa pertama
iris2 = df.iloc[50, :4].values.astype(float)  # Iris-versicolor pertama

print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"\niris1 (setosa)     = {iris1}")
print(f"iris2 (versicolor) = {iris2}")

### 3.1 Minkowski Distance

$$d_{min} = \left(\sum_{i=1}^{n} |x_i - y_i|^m\right)^{\frac{1}{m}}, \quad m \geq 1$$

In [ ]:
def minkowski_distance(x, y, m):
    return np.sum(np.abs(x - y) ** m) ** (1 / m)

for m in [1, 2, 3]:
    d = minkowski_distance(v1, v2, m)
    print(f"Minkowski (m={m}): {d:.4f}")

print("\nPada data IRIS:")
for m in [1, 2, 3]:
    d = minkowski_distance(iris1, iris2, m)
    print(f"Minkowski (m={m}): {d:.4f}")

### 3.2 Manhattan Distance (m=1)

$$d_{man} = \sum_{i=1}^{n} |x_i - y_i|$$

In [ ]:
def manhattan_distance(x, y):
    return np.sum(np.abs(x - y))

print(f"Manhattan (v1, v2)         : {manhattan_distance(v1, v2):.4f}")
print(f"Manhattan (scipy)          : {cityblock(v1, v2):.4f}")
print(f"Manhattan (iris1, iris2)    : {manhattan_distance(iris1, iris2):.4f}")

### 3.3 Euclidean Distance (m=2)

$$d_{euc} = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

In [ ]:
def euclidean_distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2))

print(f"Euclidean (v1, v2)         : {euclidean_distance(v1, v2):.4f}")
print(f"Euclidean (scipy)          : {euclidean(v1, v2):.4f}")
print(f"Euclidean (iris1, iris2)    : {euclidean_distance(iris1, iris2):.4f}")

### 3.4 Average Distance

$$d_{ave} = \left(\frac{1}{n} \sum_{i=1}^{n} (x_i - y_i)^2\right)^{\frac{1}{2}}$$

In [ ]:
def average_distance(x, y):
    n = len(x)
    return np.sqrt(np.sum((x - y) ** 2) / n)

print(f"Average Distance (v1, v2)       : {average_distance(v1, v2):.4f}")
print(f"Average Distance (iris1, iris2)  : {average_distance(iris1, iris2):.4f}")

### 3.5 Weighted Euclidean Distance

$$d_{we} = \sqrt{\sum_{i=1}^{n} w_i (x_i - y_i)^2}$$

In [ ]:
def weighted_euclidean(x, y, w):
    return np.sqrt(np.sum(w * (x - y) ** 2))

# Bobot: sepal_length lebih penting
weights = np.array([2, 1, 1, 1])
print(f"Bobot: {weights}")
print(f"Weighted Euclidean (v1, v2)       : {weighted_euclidean(v1, v2, weights):.4f}")
print(f"Weighted Euclidean (iris1, iris2)  : {weighted_euclidean(iris1, iris2, weights):.4f}")

### 3.6 Chord Distance

$$d_{chord} = \sqrt{2 - 2 \frac{\sum_{i=1}^{n} x_i y_i}{\|x\|_2 \|y\|_2}}$$

In [ ]:
def chord_distance(x, y):
    norm_x = np.linalg.norm(x)
    norm_y = np.linalg.norm(y)
    return np.sqrt(2 - 2 * np.dot(x, y) / (norm_x * norm_y))

print(f"Chord Distance (v1, v2)       : {chord_distance(v1, v2):.4f}")
print(f"Chord Distance (iris1, iris2)  : {chord_distance(iris1, iris2):.4f}")

### 3.7 Mahalanobis Distance

$$d_{mah} = \sqrt{(x - y) S^{-1} (x - y)^T}$$

Di mana $S$ adalah matriks kovariansi data.

In [ ]:
def mahalanobis_distance(x, y, S_inv):
    diff = x - y
    return np.sqrt(diff @ S_inv @ diff.T)

# Hitung matriks kovariansi dari data IRIS (4 kolom numerik)
data_numerik = df.iloc[:, :4].values
S = np.cov(data_numerik.T)
S_inv = np.linalg.inv(S)

d_mah = mahalanobis_distance(iris1, iris2, S_inv)
d_mah_scipy = mahalanobis(iris1, iris2, S_inv)

print(f"Mahalanobis (manual)  : {d_mah:.4f}")
print(f"Mahalanobis (scipy)   : {d_mah_scipy:.4f}")

### 3.8 Cosine Similarity

$$Cosine(x, y) = \frac{\sum_{i=1}^{n} x_i y_i}{\|x\|_2 \|y\|_2}$$

Nilai 1 = sangat mirip, 0 = tidak mirip.

In [ ]:
def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print(f"Cosine Similarity (v1, v2)       : {cosine_similarity(v1, v2):.4f}")
print(f"Cosine Distance   (scipy)        : {cosine(v1, v2):.4f}")
print(f"  (similarity = 1 - distance     : {1 - cosine(v1, v2):.4f})")
print(f"Cosine Similarity (iris1, iris2)  : {cosine_similarity(iris1, iris2):.4f}")

### 3.9 Pearson Correlation

$$Pearson(x, y) = \frac{\sum_{i=1}^{n}(x_i - \mu_x)(y_i - \mu_y)}{\sqrt{\sum_{i=1}^{n}(x_i - \mu_x)^2} \sqrt{\sum_{i=1}^{n}(y_i - \mu_y)^2}}$$

In [ ]:
def pearson_correlation(x, y):
    mu_x = np.mean(x)
    mu_y = np.mean(y)
    numerator = np.sum((x - mu_x) * (y - mu_y))
    denominator = np.sqrt(np.sum((x - mu_x)**2)) * np.sqrt(np.sum((y - mu_y)**2))
    return numerator / denominator

print(f"Pearson (v1, v2)       : {pearson_correlation(v1, v2):.4f}")
print(f"Pearson (scipy)        : {np.corrcoef(v1, v2)[0,1]:.4f}")
print(f"Pearson (iris1, iris2)  : {pearson_correlation(iris1, iris2):.4f}")

### 3.10 Ringkasan Semua Jarak Numerik

In [ ]:
weights = np.array([1, 1, 1, 1])

print("=" * 55)
print("RINGKASAN JARAK NUMERIK (iris1 vs iris2)")
print("=" * 55)
print(f"{'Metode':<30} {'Nilai':>10}")
print("-" * 42)
print(f"{'Manhattan':<30} {manhattan_distance(iris1, iris2):>10.4f}")
print(f"{'Euclidean':<30} {euclidean_distance(iris1, iris2):>10.4f}")
print(f"{'Minkowski (m=3)':<30} {minkowski_distance(iris1, iris2, 3):>10.4f}")
print(f"{'Average':<30} {average_distance(iris1, iris2):>10.4f}")
print(f"{'Weighted Euclidean':<30} {weighted_euclidean(iris1, iris2, weights):>10.4f}")
print(f"{'Chord':<30} {chord_distance(iris1, iris2):>10.4f}")
print(f"{'Mahalanobis':<30} {mahalanobis_distance(iris1, iris2, S_inv):>10.4f}")
print(f"{'Cosine Similarity':<30} {cosine_similarity(iris1, iris2):>10.4f}")
print(f"{'Pearson Correlation':<30} {pearson_correlation(iris1, iris2):>10.4f}")

---
## 4. Mengukur Jarak Atribut Binary

Tabel kontingensi 2x2:
- $q$: jumlah atribut = 1 untuk kedua objek
- $r$: atribut = 1 untuk objek $i$, tapi 0 untuk objek $j$
- $s$: atribut = 0 untuk objek $i$, tapi 1 untuk objek $j$
- $t$: jumlah atribut = 0 untuk kedua objek

**Symmetric binary dissimilarity:**
$$d(i,j) = \frac{r + s}{q + r + s + t}$$

**Asymmetric binary dissimilarity:**
$$d(i,j) = \frac{r + s}{q + r + s}$$

**Jaccard coefficient (similarity):**
$$sim(i,j) = \frac{q}{q + r + s}$$

In [ ]:
# Contoh data biner: fitur medis pasien
# Atribut: [Demam, Batuk, Flu, Sesak, Diare]
pasien_A = np.array([1, 1, 0, 1, 0])
pasien_B = np.array([1, 0, 1, 1, 1])

print(f"Pasien A: {pasien_A}")
print(f"Pasien B: {pasien_B}")

q = np.sum((pasien_A == 1) & (pasien_B == 1))  # keduanya 1
r = np.sum((pasien_A == 1) & (pasien_B == 0))  # A=1, B=0
s = np.sum((pasien_A == 0) & (pasien_B == 1))  # A=0, B=1
t = np.sum((pasien_A == 0) & (pasien_B == 0))  # keduanya 0

print(f"\nq (keduanya 1) = {q}")
print(f"r (A=1, B=0)   = {r}")
print(f"s (A=0, B=1)   = {s}")
print(f"t (keduanya 0) = {t}")
print(f"p (total)      = {q + r + s + t}")

# Symmetric binary dissimilarity
d_sym = (r + s) / (q + r + s + t)
print(f"\nSymmetric dissimilarity   : {d_sym:.4f}")

# Asymmetric binary dissimilarity
d_asym = (r + s) / (q + r + s)
print(f"Asymmetric dissimilarity  : {d_asym:.4f}")

# Jaccard coefficient
jaccard = q / (q + r + s)
print(f"Jaccard coefficient       : {jaccard:.4f}")
print(f"Jaccard = 1 - d_asym      : {1 - d_asym:.4f}")

---
## 5. Mengukur Jarak Tipe Categorical

### 5.1 Overlay Metric (OM)

$$d(x, y) = \sum_{i=1}^{n} \delta(a_i(x), a_i(y))$$

Di mana $\delta = 0$ jika $a_i(x) = a_i(y)$, dan $\delta = 1$ jika sebaliknya.

In [ ]:
def overlay_metric(x, y):
    """Hitung Overlay Metric antara dua objek kategorikal."""
    return sum(1 for xi, yi in zip(x, y) if xi != yi)

# Contoh: data mahasiswa [Jurusan, Gender, Angkatan]
mhs_A = ['Informatika', 'L', '2023']
mhs_B = ['Informatika', 'P', '2022']
mhs_C = ['Sistem Informasi', 'L', '2023']

print(f"Mhs A: {mhs_A}")
print(f"Mhs B: {mhs_B}")
print(f"Mhs C: {mhs_C}")

print(f"\nOM(A, B) = {overlay_metric(mhs_A, mhs_B)}")
print(f"OM(A, C) = {overlay_metric(mhs_A, mhs_C)}")
print(f"OM(B, C) = {overlay_metric(mhs_B, mhs_C)}")

### 5.2 Value Difference Metric (VDM)

$$d(x, y) = \sum_{i=1}^{n} \sum_{c=1}^{C} |P(c|a_i(x)) - P(c|a_i(y))|$$

In [ ]:
def value_difference_metric(df, obj1_idx, obj2_idx, attr_cols, class_col):
    """Hitung VDM antara dua objek dalam dataframe."""
    classes = df[class_col].unique()
    total_dist = 0

    for col in attr_cols:
        val_x = df.iloc[obj1_idx][col]
        val_y = df.iloc[obj2_idx][col]

        for c in classes:
            # P(c | a_i(x))
            mask_x = df[col] == val_x
            p_c_x = (df[mask_x][class_col] == c).sum() / mask_x.sum() if mask_x.sum() > 0 else 0

            # P(c | a_i(y))
            mask_y = df[col] == val_y
            p_c_y = (df[mask_y][class_col] == c).sum() / mask_y.sum() if mask_y.sum() > 0 else 0

            total_dist += abs(p_c_x - p_c_y)

    return total_dist

# Contoh dengan data kategorikal + kelas
df_cat = pd.DataFrame({
    'Warna': ['Merah', 'Biru', 'Merah', 'Hijau', 'Biru', 'Hijau'],
    'Bentuk': ['Bulat', 'Kotak', 'Bulat', 'Kotak', 'Bulat', 'Kotak'],
    'Kelas': ['A', 'B', 'A', 'B', 'A', 'B']
})
print(df_cat)
print()

vdm = value_difference_metric(df_cat, 0, 1, ['Warna', 'Bentuk'], 'Kelas')
print(f"VDM antara objek 0 dan 1: {vdm:.4f}")

vdm2 = value_difference_metric(df_cat, 0, 2, ['Warna', 'Bentuk'], 'Kelas')
print(f"VDM antara objek 0 dan 2: {vdm2:.4f}")

### 5.3 Minimum Risk Metric (MRM)

$$d(x, y) = \sum_{c=1}^{C} P(c|x)(1 - P(c|y))$$

In [ ]:
def minimum_risk_metric(df, obj1_idx, obj2_idx, attr_cols, class_col):
    """Hitung MRM antara dua objek."""
    classes = df[class_col].unique()

    # Estimasi P(c|x) menggunakan Naive Bayes sederhana
    def estimate_p_c(obj_idx):
        probs = {}
        for c in classes:
            p = 1.0
            for col in attr_cols:
                val = df.iloc[obj_idx][col]
                mask = df[class_col] == c
                p *= (df[mask][col] == val).sum() / mask.sum() if mask.sum() > 0 else 0
            probs[c] = p
        # Normalisasi
        total = sum(probs.values())
        if total > 0:
            probs = {c: p / total for c, p in probs.items()}
        return probs

    p_x = estimate_p_c(obj1_idx)
    p_y = estimate_p_c(obj2_idx)

    mrm = sum(p_x[c] * (1 - p_y[c]) for c in classes)
    return mrm

mrm = minimum_risk_metric(df_cat, 0, 1, ['Warna', 'Bentuk'], 'Kelas')
print(f"MRM antara objek 0 dan 1: {mrm:.4f}")

mrm2 = minimum_risk_metric(df_cat, 0, 2, ['Warna', 'Bentuk'], 'Kelas')
print(f"MRM antara objek 0 dan 2: {mrm2:.4f}")

---
## 6. Mengukur Jarak Tipe Ordinal

Langkah:
1. Ganti nilai ordinal dengan ranking $r_{if} \in \{1, ..., M_f\}$
2. Normalisasi: $z_{if} = \frac{r_{if} - 1}{M_f - 1}$
3. Hitung jarak dengan metode numerik biasa

In [ ]:
# Contoh data ordinal
df_ord = pd.DataFrame({
    'Ukuran': ['Kecil', 'Besar', 'Sedang', 'Kecil', 'Besar'],
    'Kepuasan': ['Puas', 'Sangat Puas', 'Netral', 'Tidak Puas', 'Puas']
})

# Mapping ordinal ke ranking
ukuran_map = {'Kecil': 1, 'Sedang': 2, 'Besar': 3}
kepuasan_map = {'Sangat Tidak Puas': 1, 'Tidak Puas': 2, 'Netral': 3, 'Puas': 4, 'Sangat Puas': 5}

df_ord['Ukuran_rank'] = df_ord['Ukuran'].map(ukuran_map)
df_ord['Kepuasan_rank'] = df_ord['Kepuasan'].map(kepuasan_map)

# Normalisasi z = (r - 1) / (M - 1)
M_ukuran = 3
M_kepuasan = 5

df_ord['Ukuran_z'] = (df_ord['Ukuran_rank'] - 1) / (M_ukuran - 1)
df_ord['Kepuasan_z'] = (df_ord['Kepuasan_rank'] - 1) / (M_kepuasan - 1)

print(df_ord)

# Hitung jarak Euclidean antara objek 0 dan 1 menggunakan nilai z
z_cols = ['Ukuran_z', 'Kepuasan_z']
obj0 = df_ord.loc[0, z_cols].values.astype(float)
obj1 = df_ord.loc[1, z_cols].values.astype(float)

d = euclidean_distance(obj0, obj1)
print(f"\nObjek 0: {df_ord.loc[0, ['Ukuran', 'Kepuasan']].values} -> z = {obj0}")
print(f"Objek 1: {df_ord.loc[1, ['Ukuran', 'Kepuasan']].values} -> z = {obj1}")
print(f"Euclidean distance (ordinal): {d:.4f}")

---
## 7. Menghitung Jarak Tipe Campuran

Untuk data dengan atribut campuran (nominal, binary, numerik, ordinal):

$$d(i,j) = \frac{\sum_{f=1}^{p} \delta_{ij}^{(f)} d_{ij}^{(f)}}{\sum_{f=1}^{p} \delta_{ij}^{(f)}}$$

- Jika $f$ numerik: $d_{ij}^{f} = \frac{|x_{if} - x_{jf}|}{\max_h x_{hf} - \min_h x_{hf}}$
- Jika $f$ nominal/binary: $d_{ij}^{f} = 0$ jika $x_{if} = x_{jf}$, else $1$
- Jika $f$ ordinal: transformasi ke $z_{if}$, lalu perlakukan seperti numerik

In [ ]:
# Data campuran
df_mix = pd.DataFrame({
    'Nama': ['Ali', 'Budi', 'Cici', 'Dina'],
    'Usia': [25, 30, 22, 35],                         # Numerik
    'Gaji': [5000, 8000, 4500, 9000],                 # Numerik
    'Gender': ['L', 'L', 'P', 'P'],                   # Nominal
    'Pendidikan': ['S1', 'S2', 'SMA', 'S2'],          # Ordinal
    'Merokok': [1, 0, 0, 1],                          # Binary
})
print(df_mix)

# Konfigurasi tipe atribut
attr_config = {
    'Usia': 'numerik',
    'Gaji': 'numerik',
    'Gender': 'nominal',
    'Pendidikan': 'ordinal',
    'Merokok': 'binary_asym',
}

ordinal_maps = {
    'Pendidikan': {'SMA': 1, 'S1': 2, 'S2': 3}
}

def mixed_distance(df, i, j, config, ord_maps):
    numerator = 0
    denominator = 0

    for attr, tipe in config.items():
        xi = df.iloc[i][attr]
        xj = df.iloc[j][attr]

        # Cek missing value
        if pd.isna(xi) or pd.isna(xj):
            continue

        # Binary asymmetric: skip jika keduanya 0
        if tipe == 'binary_asym' and xi == 0 and xj == 0:
            continue

        delta = 1  # kontribusi atribut

        if tipe == 'numerik':
            range_val = df[attr].max() - df[attr].min()
            d_f = abs(xi - xj) / range_val if range_val > 0 else 0
        elif tipe in ('nominal', 'binary_sym', 'binary_asym'):
            d_f = 0 if xi == xj else 1
        elif tipe == 'ordinal':
            mapping = ord_maps[attr]
            M = len(mapping)
            z_i = (mapping[xi] - 1) / (M - 1)
            z_j = (mapping[xj] - 1) / (M - 1)
            range_z = 1.0  # sudah dinormalisasi ke [0,1]
            d_f = abs(z_i - z_j) / range_z

        numerator += delta * d_f
        denominator += delta

    return numerator / denominator if denominator > 0 else 0

# Hitung jarak campuran antara semua pasangan
n = len(df_mix)
print("\nMatriks Jarak Campuran:")
print(f"{'':>8}", end='')
for i in range(n):
    print(f"{df_mix.iloc[i]['Nama']:>8}", end='')
print()

for i in range(n):
    print(f"{df_mix.iloc[i]['Nama']:>8}", end='')
    for j in range(n):
        d = mixed_distance(df_mix, i, j, attr_config, ordinal_maps)
        print(f"{d:>8.4f}", end='')
    print()

---
## Kesimpulan

Pada notebook ini telah diimplementasikan:

1. **Distribusi Data** - Visualisasi distribusi normal dan histogram data IRIS
2. **Statistik Deskriptif** - Mean, Median, Modus, Range, IQR, Variansi, Standar Deviasi, Skewness, Boxplot
3. **Jarak Numerik** - Minkowski, Manhattan, Euclidean, Average, Weighted Euclidean, Chord, Mahalanobis, Cosine, Pearson
4. **Jarak Binary** - Symmetric/Asymmetric dissimilarity, Jaccard coefficient
5. **Jarak Categorical** - Overlay Metric (OM), Value Difference Metric (VDM), Minimum Risk Metric (MRM)
6. **Jarak Ordinal** - Normalisasi ranking ke [0, 1]
7. **Jarak Campuran** - Menggabungkan semua tipe atribut dalam satu formula